# QuantView Notebook Charts Demo

This notebook demonstrates all the charting capabilities of the QuantView notebook module.

In [ ]:
# Setup
import sys

sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import display

from quantview.notebook import (
    NSSFit,
    YieldCurveData,
    candlestick_chart,
    create_chart_dashboard,
    create_equity_dashboard_controls,
    create_output_area,
    create_vol_surface_controls,
    create_yield_curve_controls,
    export_html,
    options_chain_greeks,
    vol_surface_2d,
    vol_surface_3d,
    yield_curve_carry_roll,
    yield_curve_chart,
)
from quantview.notebook.export import export_dashboard

# Set default renderer for notebook
pio.renderers.default = "notebook"

print("QuantView Notebook Charts Demo")
print("=" * 50)

## 1. Candlestick Chart with Technical Indicators

In [ ]:
# Generate synthetic OHLCV data
np.random.seed(42)
dates = pd.date_range("2024-01-01", periods=252, freq="B")
returns = np.random.normal(0.0005, 0.015, len(dates))
prices = 100 * np.exp(np.cumsum(returns))

noise = np.random.uniform(0.995, 1.005, (len(dates), 4))
df = pd.DataFrame(
    {
        "open": prices * noise[:, 0],
        "high": prices * noise[:, 1],
        "low": prices * noise[:, 2],
        "close": prices * noise[:, 3],
        "volume": np.random.lognormal(13, 0.5, len(dates)).astype(int),
    },
    index=dates,
)

# Ensure high/low are correct
df["high"] = df[["open", "high", "close"]].max(axis=1)
df["low"] = df[["open", "low", "close"]].min(axis=1)

print(f"Data shape: {df.shape}")
print(f"Date range: {df.index[0].date()} to {df.index[-1].date()}")
df.tail()

In [ ]:
# Basic candlestick chart
fig = candlestick_chart(df, "SPY", indicators=["sma_20", "sma_50", "bbands"])
fig.show()

In [ ]:
# Full chart with all indicators
fig_full = candlestick_chart(
    df,
    "SPY",
    indicators=["sma_20", "sma_50", "sma_200", "ema_12", "ema_26", "bbands", "rsi", "macd"],
    volume=True,
    height=900,
)
fig_full.show()

## 2. Volatility Surface Charts

Requires the `black_scholes` package for SVI fitting.

In [ ]:
try:
    from black_scholes import OptionParams, black_scholes_price
    from black_scholes.surface import VolSurface

    HAS_BS = True
    print("black_scholes available - vol surface demos enabled")
except ImportError:
    HAS_BS = False
    print("black_scholes not available - skipping vol surface demos")

In [ ]:
if HAS_BS:
    # Create synthetic vol surface
    strikes = np.array([70, 80, 90, 95, 100, 105, 110, 120, 130, 140, 150])
    vols = np.array([0.32, 0.28, 0.25, 0.235, 0.23, 0.235, 0.24, 0.26, 0.29, 0.31, 0.33])
    spot = 100.0
    T = 0.25
    r = 0.05

    prices = [
        black_scholes_price(OptionParams(spot, k, T, r, v))
        for k, v in zip(strikes, vols, strict=True)
    ]
    surface = VolSurface.from_market_prices(spot, strikes, prices, T, r)
    svi_fit = surface.fit()

    print(f"Surface: {len(surface.strikes)} strikes, T={T}Y")
    print(f"Forward: {surface.forward:.2f}")
    print(f"SVI RMS Error: {svi_fit.rms_error:.6f}")
    print(f"SVI Max Error: {svi_fit.max_error:.6f}")

In [ ]:
if HAS_BS:
    # 3D Volatility Surface
    fig_3d = vol_surface_3d(surface, svi_fit, title="SPX Volatility Surface (3M)", height=700)
    fig_3d.show()

In [ ]:
if HAS_BS:
    # 2D Vol Smile with SVI and Residuals
    fig_2d = vol_surface_2d(surface, svi_fit, title="SPX Vol Smile (3M)", height=600)
    fig_2d.show()

## 3. Yield Curve Charts

In [ ]:
# Sample yield curve data
tenors = np.array([0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30])
spot = np.array([0.052, 0.050, 0.048, 0.046, 0.045, 0.044, 0.044, 0.0445, 0.0455, 0.046])
forward = np.array([0.052, 0.049, 0.046, 0.044, 0.0435, 0.043, 0.0435, 0.0445, 0.046, 0.047])
par = np.array([0.052, 0.050, 0.048, 0.046, 0.045, 0.044, 0.044, 0.0445, 0.0455, 0.046])

curve = YieldCurveData(tenors=tenors, spot_rates=spot, forward_rates=forward, par_rates=par)

# NSS fit (rough calibration)
nss = NSSFit(beta0=0.045, beta1=-0.01, beta2=-0.015, beta3=0.005, tau1=1.5, tau2=8.0)

print("Yield Curve Data:")
for t, s, f, p in zip(tenors, spot, forward, par, strict=True):
    print(f"  {t:>5.2f}Y: Spot={s:.3%} Forward={f:.3%} Par={p:.3%}")

In [ ]:
# Yield curve with NSS fit
fig_yc = yield_curve_chart(curve, nss_fit=nss, title="USD Treasury Yield Curve", height=600)
fig_yc.show()

In [ ]:
# Carry & Roll-Down
fig_cr = yield_curve_carry_roll(
    tenors,
    spot,
    forward,
    horizon=1.0,
    title="Carry & Roll-Down (1Y Horizon)",
    height=500,
)
fig_cr.show()

## 4. Options Chain Greeks

Interactive table with all Greeks for calls and puts.

In [ ]:
if HAS_BS:
    strikes = np.arange(80, 131, 5)
    vols = 0.23 + 0.1 * (strikes - 100) ** 2 / 10000  # Simple smile
    surface = VolSurface(
        spot=100.0,
        strikes=tuple(strikes),
        implied_vols=tuple(vols),
        time_to_maturity=0.25,
        risk_free_rate=0.05,
        dividend_yield=0.0,
    )

    fig_opt = options_chain_greeks(
        surface, expiry=0.25, title="SPX Options Chain (3M Expiry)", height=700
    )
    fig_opt.show()

## 5. Interactive Widgets Demo

In [ ]:
# Create dashboard controls
equity_controls = create_equity_dashboard_controls(["SPY", "QQQ", "AAPL", "MSFT", "GOOGL"])
vol_controls = create_vol_surface_controls()
yield_controls = create_yield_curve_controls()
output = create_output_area()

# Display controls

display(equity_controls)
display(vol_controls)
display(yield_controls)

In [ ]:
# Full dashboard layout
dashboard = create_chart_dashboard(equity_controls, vol_controls, yield_controls, output)
display(dashboard)

## 6. Export Charts

In [ ]:
# Export to HTML
export_html(fig, "demo_candlestick.html")
print("Saved demo_candlestick.html")

if HAS_BS:
    export_html(fig_3d, "demo_vol_3d.html")
    export_html(fig_2d, "demo_vol_2d.html")
    export_html(fig_opt, "demo_options_chain.html")
    print("Saved vol surface and options chain HTML")

export_html(fig_yc, "demo_yield_curve.html")
export_html(fig_cr, "demo_carry_roll.html")
print("Saved yield curve HTML")

In [ ]:
# Batch export dashboard

figures = {
    "SPY Candlestick": fig,
    "SPY Full Indicators": fig_full,
    "Yield Curve": fig_yc,
    "Carry & Roll": fig_cr,
}
if HAS_BS:
    figures["Vol Surface 3D"] = fig_3d
    figures["Vol Smile 2D"] = fig_2d
    figures["Options Chain"] = fig_opt

export_dashboard(figures, "demo_dashboard", prefix="chart", formats=["html", "png"])
print("Dashboard exported to demo_dashboard/")

## 7. SVI Parameter Exploration